In [ ]:
!pip install tensorflow numpy h5py scikit-learn matplotlib

In [6]:
import os
import numpy as np
import h5py
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, accuracy_score

In [7]:

def load_h5_data(directory, is_training=True, target_size=(256, 256), max_samples=None):
    images, masks = [], []
    sample_count = 0
    
    for filename in os.listdir(directory):
        if max_samples and sample_count >= max_samples:
            break
        if filename.endswith('.h5'):
            try:
                with h5py.File(os.path.join(directory, filename), 'r') as f:
                    image = f['image'][:]  # Shape: (H, W) or (slices, H, W)
                    if is_training:
                        mask = f['label'][:]  # Shape: (H, W) or (slices, H, W)
                    
                    # Handle 3D volumes (test data) or 2D slices (training data)
                    if image.ndim == 3:  # (slices, H, W)
                        for slice_idx in range(image.shape[0]):
                            slice_img = image[slice_idx]  # (H, W)
                            slice_img = np.expand_dims(slice_img, axis=-1)  # (H, W, 1)
                            slice_img = tf.image.resize(slice_img, target_size, method='bilinear').numpy()
                            images.append(slice_img)
                            if is_training:
                                slice_mask = mask[slice_idx]
                                slice_mask = np.expand_dims(slice_mask, axis=-1)
                                slice_mask = tf.image.resize(slice_mask, target_size, method='nearest').numpy()
                                masks.append(np.squeeze(slice_mask, axis=-1))
                    elif image.ndim == 2:  # (H, W)
                        image = np.expand_dims(image, axis=-1)  # (H, W, 1)
                        image = tf.image.resize(image, target_size, method='bilinear').numpy()
                        images.append(image)
                        if is_training:
                            mask = np.expand_dims(mask, axis=-1)
                            mask = tf.image.resize(mask, target_size, method='nearest').numpy()
                            masks.append(np.squeeze(mask, axis=-1))
                    else:
                        print(f"Unexpected shape in {filename}: {image.shape}")
                        continue
                    
                    sample_count += 1
            except Exception as e:
                print(f"Error loading {filename}: {e}")
                continue
    
    images = np.array(images, dtype=np.float32)
    masks = np.array(masks, dtype=np.uint8) if is_training else None
    return images, masks

In [8]:

train_dir = '/kaggle/input/acdc-dataset/ACDC_preprocessed/ACDC_training_slices'
train_images, train_masks = load_h5_data(train_dir, is_training=True, target_size=(256, 256), max_samples=1500)

# Normalize images to [0, 1]
train_images = train_images / np.max(train_images)

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_images, train_masks, test_size=0.2, random_state=42)

# Load test data
test_dir = '/kaggle/input/acdc-dataset/ACDC_preprocessed/ACDC_testing_volumes'
test_images, _ = load_h5_data(test_dir, is_training=False, target_size=(256, 256), max_samples=100)  # Reduced for memory

# Normalize test images
test_images = test_images / np.max(test_images)

print(f"Training shape: {X_train.shape}, Validation shape: {X_val.shape}, Test shape: {test_images.shape}")

Training shape: (1200, 256, 256, 1), Validation shape: (300, 256, 256, 1), Test shape: (1076, 256, 256, 1)


In [9]:

def duck_net(input_shape, num_classes=4):
    inputs = layers.Input(input_shape)
    
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        return x
    
    # Downsampling path
    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    c4 = conv_block(p3, 256)
    
    # Bottleneck
    bottleneck = conv_block(c4, 512)
    
    # Upsampling path with skip connections
    u4 = layers.UpSampling2D((2, 2))(bottleneck)
    u4 = layers.Concatenate()([u4, c3])
    c5 = conv_block(u4, 256)
    u3 = layers.UpSampling2D((2, 2))(c5)
    u3 = layers.Concatenate()([u3, c2])
    c6 = conv_block(u3, 128)
    u2 = layers.UpSampling2D((2, 2))(c6)
    u2 = layers.Concatenate()([u2, c1])
    c7 = conv_block(u2, 64)
    
    # Output layer
    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(c7)
    
    model = models.Model(inputs, outputs)
    return model

In [10]:
# Set input shape
input_shape = (256, 256, 1)
model = duck_net(input_shape, num_classes=4)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 256, 256, 1)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d (Conv2D)           │ (None, 256, 256, 32)   │            320 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 256, 256, 32)   │            128 │ conv2d[0][0]           │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_1 (Conv2D)         │ (None, 256, 256, 32)   │          9,248 │ batch_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 256, 256, 32)   │            128 │ conv2d_1[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d             │ (None, 128, 128, 32)   │              0 │ batch_normalization_1… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_2 (Conv2D)         │ (None, 128, 128, 64)   │         18,496 │ max_pooling2d[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_2     │ (None, 128, 128, 64)   │            256 │ conv2d_2[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_3 (Conv2D)         │ (None, 128, 128, 64)   │         36,928 │ batch_normalization_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_3     │ (None, 128, 128, 64)   │            256 │ conv2d_3[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_1           │ (None, 64, 64, 64)     │              0 │ batch_normalization_3… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_4 (Conv2D)         │ (None, 64, 64, 128)    │         73,856 │ max_pooling2d_1[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_4     │ (None, 64, 64, 128)    │            512 │ conv2d_4[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_5 (Conv2D)         │ (None, 64, 64, 128)    │        147,584 │ batch_normalization_4… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_5     │ (None, 64, 64, 128)    │            512 │ conv2d_5[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_2      

 Total params: 7,433,828 (28.36 MB)

 Trainable params: 7,428,068 (28.34 MB)

 Non-trainable params: 5,760 (22.50 KB)

In [ ]:
# --- Step 3: Training the Model ---
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=8)

Epoch 1/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 3334s 22s/step - accuracy: 0.7773 - loss: 0.9902 - val_accuracy: 0.9022 - val_loss: 0.3656
Epoch 2/15
  2/150 ━━━━━━━━━━━━━━━━━━━━ 50:32 20s/step - accuracy: 0.9777 - loss: 0.1441

In [ ]:
# Save model weights after training
model.save_weights("model_weights.weights.h5")
print("Model weights saved to model_weights.weights.h5")


In [ ]:
model.save("trained_model.keras")

In [ ]:
from tensorflow.keras.models import load_model

# Load the saved model
loaded_model = load_model("trained_model.keras")

# Verify the model architecture
loaded_model.summary()

In [ ]:
# --- Step 4: Evaluation Metrics ---
def compute_metrics(y_true, y_pred):
    y_true_flat = y_true.flatten()
    y_pred_flat = np.argmax(y_pred, axis=-1).flatten()
    
    dice = 2 * np.sum(y_true_flat * y_pred_flat) / (np.sum(y_true_flat) + np.sum(y_pred_flat) + 1e-6)
    jaccard = np.sum(y_true_flat * y_pred_flat) / (np.sum(y_true_flat) + np.sum(y_pred_flat) - np.sum(y_true_flat * y_pred_flat) + 1e-6)
    precision = precision_score(y_true_flat, y_pred_flat, average='macro', zero_division=0)
    recall = recall_score(y_true_flat, y_pred_flat, average='macro', zero_division=0)
    accuracy = accuracy_score(y_true_flat, y_pred_flat)
    
    return dice, jaccard, precision, recall, accuracy

# Predict on validation and test sets
val_preds = model.predict(X_val)
test_preds = model.predict(test_images)

# Compute metrics on validation set
dice, jaccard, precision, recall, accuracy = compute_metrics(y_val, val_preds)
print(f"Validation Metrics - Dice: {dice:.4f}, Jaccard: {jaccard:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
import pandas as pd

# Save test predictions to CSV
predicted_classes = np.argmax(test_preds, axis=-1)  # Shape: (N, H, W)

rows = []
for i in range(len(test_images)):
    # Flatten spatial dims; store per-sample summary stats
    pred_flat = predicted_classes[i].flatten()
    rows.append({
        "sample_index": i,
        "predicted_class_mode": int(np.bincount(pred_flat).argmax()),  # dominant class
        "pct_background": float(np.mean(pred_flat == 0)),
        "pct_LV":         float(np.mean(pred_flat == 1)),
        "pct_RV":         float(np.mean(pred_flat == 2)),
        "pct_MYO":        float(np.mean(pred_flat == 3)),
    })

test_results_df = pd.DataFrame(rows)
test_results_df.to_csv("test_predictions.csv", index=False)
print(f"Test predictions saved to test_predictions.csv ({len(test_results_df)} samples)")
test_results_df.head()


In [2]:
# --- Step 5: Visualization ---
# Validation Set Visualization (with ground truth)
print("Validation Set Visualization (Ground Truth vs Predicted):")
for i in range(30):
    plt.figure(figsize=(20, 5))
    plt.subplot(1, 4, 1)
    plt.title("Input Image")
    plt.imshow(X_val[i, :, :, 0], cmap='gray')
    plt.axis('off')
    plt.subplot(1, 4, 2)
    plt.title("Ground Truth Mask")
    plt.imshow(y_val[i], cmap='jet', vmin=0, vmax=3)  # 0: background, 1: LV, 2: RV, 3: MYO
    plt.axis('off')
    plt.subplot(1, 4, 3)
    plt.title("Predicted Mask")
    plt.imshow(np.argmax(val_preds[i], axis=-1), cmap='jet', vmin=0, vmax=3)
    plt.axis('off')
    plt.subplot(1, 4, 4)
    plt.title("Overlay (Predicted)")
    plt.imshow(X_val[i, :, :, 0], cmap='gray')
    plt.imshow(np.argmax(val_preds[i], axis=-1), cmap='jet', alpha=0.5, vmin=0, vmax=3)
    plt.axis('off')
    plt.show()

# Test Set Visualization (no ground truth)
""" 
print("Test Set Visualization (Predicted Only):")
for i in range(10):
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.title("Input Image")
    plt.imshow(test_images[i, :, :, 0], cmap='gray')
    plt.axis('off')
    plt.subplot(1, 3, 2)
    plt.title("Predicted Mask")
    plt.imshow(np.argmax(test_preds[i], axis=-1), cmap='jet', vmin=0, vmax=3)
    plt.axis('off')
    plt.subplot(1, 3, 3)
    plt.title("Overlay")
    plt.imshow(test_images[i, :, :, 0], cmap='gray')
    plt.imshow(np.argmax(test_preds[i], axis=-1), cmap='jet', alpha=0.5, vmin=0, vmax=3)
    plt.axis('off')
    plt.show()
"""

Validation Set Visualization (Ground Truth vs Predicted):


NameError: name 'plt' is not defined

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K

K.clear_session()

model = unet_classifier(
    input_shape=(256, 256, 1),
    num_classes=4,
    base_filters=32,
    dropout_rate=0.3,
    dense_units=256,
    learning_rate=1e-4
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
    run_eagerly=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=8
)

In [ ]:
import h5py
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "/kaggle/input/datasets/abdulraufchodhry/acdc-severity/acdc_severity.csv"

df = pd.read_csv(csv_path)

severity_map = {
    0: "normal",
    1: "mild",
    2: "moderate",
    3: "severe"
}

sample_df = df.head(5)

for row in sample_df.itertuples():
    with h5py.File(row.image_path, "r") as h5_file:
        image = h5_file["image"][:]

    severity_number = int(row.severity_number)
    severity_type = severity_map[severity_number]

    plt.figure(figsize=(6, 6))
    plt.imshow(image, cmap="gray")
    plt.title(
        f"{row.image_name}\n"
        f"Severity Number: {severity_number}\n"
        f"Severity Type: {severity_type} \n",
        fontsize=10
    )
    plt.axis("off")
    plt.show()

    print("\n")